# Task 5 - Judge Model

In the previous tasks, product descriptions were evaluated manually - accurate but slow and impossible to scale. This notebook builds an automated judge: an LLM that applies the same rubric from Task 1 to grade any description, returning structured verdicts with explanations.

> **Deliverable:** this notebook with the judge prompt, Pydantic schema, and a sanity-check run on 5 products.

## Model Choice

Task 2 used `meta-llama/Meta-Llama-3.1-8B-Instruct`. The assignment recommends starting the judge with the model not used in Task 2 (`google/gemma-2-9b-it`), then switching to a larger model if needed.

**Judge model chosen: `Qwen/Qwen3-30B-A3B-Instruct-2507`**

`google/gemma-2-9b-it-fast` (the closest available variant on Nebius - the base model is not listed) was tested first in `task5_judge_first_try_gemma.ipynb` using the identical Pydantic schema and judge prompt. Two runs were conducted:

- **`max_tokens=1024`** — 1 out of 5 calls failed (Garmin, token limit). Nintendo Switch received `length=bad` for 84 words, which the rubric defines as `good`.
- **`max_tokens=2048`** — Garmin still failed. Sony and Nintendo Switch both received wrong length verdicts (`ok` for 80–82 words, should be `good`). Latency reached 8 728 ms.

The model miscount words inconsistently and fails to complete structured JSON for longer prompts regardless of token budget. These failures justify switching to a more capable model.

`Qwen/Qwen3-30B-A3B-Instruct-2507` was chosen for three reasons:

- It is from a different architecture family than the Llama generator, reducing the risk of shared biases between generator and judge.
- The 30B Mixture-of-Experts design (activating ~3B parameters per token) provides strong instruction-following at competitive latency and cost.
- It is explicitly listed in the assignment as the recommended fallback for judge tasks.

## Imports & Configuration

In [1]:
import os
import sys
import time
import json
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from openai import OpenAI

API_KEY      = os.getenv("NEBIUS_API_KEY")
BASE_URL     = "https://api.tokenfactory.nebius.com/v1/"
JUDGE_MODEL  = "Qwen/Qwen3-30B-A3B-Instruct-2507"

XLSX_PATH    = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01.xlsx")
JUDGE_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]

## Output Schema

### Why Pydantic?

Without a structured output contract, the judge can return verdicts in any format - sometimes a JSON object, sometimes plain text, sometimes a different key name. Pydantic provides a schema that the API enforces at generation time (not just parsed after the fact), so every response is guaranteed to have the right fields and types. This makes the judge output directly usable in downstream code without fragile string parsing.

### Why does explanation come before verdict?

The ordering is deliberate and has a direct effect on output quality. When the model generates the **explanation first**, it is forced to reason through the evidence, read the description, compare it against the criterion definition, identify what works and what does not, before it commits to a verdict. The verdict then follows naturally from that reasoning.

If the **verdict came first**, the model would pick a label and then construct a post-hoc justification to support it. The explanation would be rationalisation rather than reasoning, and borderline cases would be judged less carefully.

This is the same principle behind chain-of-thought prompting: making the model *show its work before answering* consistently improves accuracy, especially on nuanced evaluation tasks where the right verdict depends on weighing multiple factors.

In [2]:
class CriterionRating(BaseModel):
    explanation: str          # reasoning comes first — forces the model to analyse before concluding
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    fluency:   CriterionRating
    grammar:   CriterionRating
    tone:      CriterionRating
    length:    CriterionRating
    grounding: CriterionRating

## Judge Prompt

The prompt embeds the full rubric definitions so the judge applies the same standards used during manual evaluation in Task 3. Cost and latency are excluded - those are measured programmatically and do not require LLM judgment.

**Grounding** requires special attention: the judge must see the original product data (name, attributes, material, warranty) to verify whether each claim in the description is supported. Without this context, the judge cannot distinguish a grounded fact from an invented one. The user message therefore always includes both the product data and the generated description side by side.

In [3]:
JUDGE_SYSTEM_PROMPT = """\
You are an expert evaluator of e-commerce product descriptions. \
Your task is to rate a generated product description against five quality criteria.

For each criterion, you must provide:
  1. explanation — your reasoning (analyse the text, cite specific phrases if relevant)
  2. verdict     — one of: good | ok | bad

Always write the explanation before the verdict. Do not pick a verdict first and then justify it.

=== RUBRIC ===

FLUENCY
  good : The description reads naturally and engagingly. Sentences flow well, transitions are smooth, \
and the writing feels polished.
  ok   : Readable overall, but contains minor awkward phrases, repetition, or slightly choppy transitions \
that interrupt the flow.
  bad  : Difficult to read. Contains confusing structure, very unnatural phrasing, or reads like a \
rough draft.

GRAMMAR
  good : No spelling, punctuation, or grammatical errors.
  ok   : One or two minor errors (typo, missing comma, minor agreement issue) that do not impede \
understanding.
  bad  : Multiple errors, or errors that make the text confusing or unprofessional.

TONE
  good : Warm, confident, and benefit-focused. Leads with what the customer gains. Avoids dry spec \
lists and hollow hype words ("amazing", "revolutionary").
  ok   : Adequate but imperfect — too neutral/dry, or slightly over-hyped, or mixes benefit-focus \
with spec-list language.
  bad  : Inappropriate register — cold, arrogant, sarcastic, or wildly mismatched to a retail context.

LENGTH
  good : Between 50 and 90 words (inclusive).
  ok   : Between 40–49 words or 91–110 words.
  bad  : 39 words or fewer, or 111 words or more.
Count words carefully before assigning a verdict.

GROUNDING
  good : Every factual claim in the description is directly supported by the product information provided.
  ok   : The description makes a minor reasonable inference not explicitly stated but plausible given \
the product data.
  bad  : The description invents at least one feature, material, specification, or fact that is NOT \
present in the product information.
IMPORTANT: Marketing language ("lightning-fast", "stunning", "powerhouse", "breathtaking") applied \
to real, listed features is NOT a grounding failure. Only penalise fabricated facts.
"""


def build_judge_message(row: pd.Series) -> str:
    return (
        "=== PRODUCT INFORMATION ===\n"
        f"Product name : {row['product_name']}\n"
        f"Attributes   : {row['Product_attribute_list']}\n"
        f"Material     : {row['material']}\n"
        f"Warranty     : {row['warranty']}\n"
        "\n"
        "=== GENERATED DESCRIPTION ===\n"
        f"{row['generated_description']}\n"
        "\n"
        "Evaluate the description against all five criteria (fluency, grammar, tone, length, grounding). "
        "For each criterion write your explanation first, then your verdict."
    )

## Judge Function

In [4]:
def judge_description(client: OpenAI, row: pd.Series) -> dict:
    """
    Call the judge model for one product row.
    Returns a flat dict with keys like fluency_explanation, fluency_verdict, etc.
    """
    try:
        response = client.beta.chat.completions.parse(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user",   "content": build_judge_message(row)},
            ],
            response_format=JudgeOutput,
            temperature=0.1,   # low temperature for consistent, reproducible judgements
            max_tokens=1024,
        )
        result: JudgeOutput = response.choices[0].message.parsed
        flat = {}
        for criterion in JUDGE_CRITERIA:
            rating = getattr(result, criterion)
            flat[f"{criterion}_explanation"] = rating.explanation
            flat[f"{criterion}_verdict"]     = rating.verdict
        flat["judge_error"] = ""
        return flat
    except Exception as e:
        flat = {}
        for criterion in JUDGE_CRITERIA:
            flat[f"{criterion}_explanation"] = ""
            flat[f"{criterion}_verdict"]     = ""
        flat["judge_error"] = str(e)
        return flat

## Sanity Check = 5 Products

Before running the full dataset, we test the judge on 5 products from the baseline sheet to verify that:
- The structured output schema is respected
- Verdicts are reasonable and match the human evaluation direction
- Explanations are specific (citing actual phrases) rather than generic

In [5]:
SANITY_ROWS = [0, 3, 9, 13, 14]   # iPhone, Sony headphones, Garmin, Nintendo Switch, PS5

df_baseline = pd.read_excel(XLSX_PATH, sheet_name="baseline")
client      = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print("=" * 70)
print("TASK 5 — Judge Sanity Check (5 products)")
print("=" * 70)
print(f"Judge model: {JUDGE_MODEL}\n")

for idx in SANITY_ROWS:
    row = df_baseline.iloc[idx]
    print(f"[{idx:02d}] {row['product_name']}")
    start  = time.time()
    result = judge_description(client, row)
    elapsed = round((time.time() - start) * 1000)

    if result["judge_error"]:
        print(f"  ERROR: {result['judge_error']}\n")
        continue

    for criterion in JUDGE_CRITERIA:
        v   = result[f"{criterion}_verdict"]
        exp = result[f"{criterion}_explanation"]
        print(f"  {criterion:<10} [{v}]  {exp[:110]}")
    print(f"  latency: {elapsed} ms\n")

TASK 5 — Judge Sanity Check (5 products)
Judge model: Qwen/Qwen3-30B-A3B-Instruct-2507

[00] Apple iPhone 15 Pro
  fluency    [good]  The description flows naturally with smooth transitions between sentences. Phrases like 'Unleash peak performa
  grammar    [good]  The description contains no spelling, punctuation, or grammatical errors. All sentences are properly construct
  tone       [good]  The tone is warm, confident, and benefit-focused, emphasizing user experience rather than just listing specs. 
  length     [good]  The description contains exactly 87 words. This falls within the ideal range of 50 to 90 words as specified in
  grounding  [good]  Every claim in the description is directly supported by the provided product information. The A17 Pro chip is 
  latency: 5954 ms

[03] Sony WH‑1000XM5 Headphones
  fluency    [good]  The description flows naturally with smooth transitions between ideas. Sentences connect logically—starting wi
  grammar    [good]  There are no spelling,